### Importing Packages

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

### Reading Minute Level Data

In [2]:
minute_2020_data = pd.read_csv('D:/PhD EMP/2020/minute_2020_data.csv').drop('Unnamed: 0', axis=1)

### Preprocessing Minute Level Dataset

In [3]:
minute_2020_data["team"] = (
    minute_2020_data["player_name"].astype(str)
    .str.split("-", n=1)
    .str[0]
)

# Ensure datetime types
minute_2020_data['date'] = pd.to_datetime(minute_2020_data['date'])
minute_2020_data['minute'] = pd.to_datetime(minute_2020_data['minute'])

# Replace date part of minute with the real date
minute_2020_data['minute'] = minute_2020_data['date'] + (minute_2020_data['minute'] - minute_2020_data['minute'].dt.normalize())

### Final Minute Level Modelling Dataset

In [4]:
minute_2020_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 352287 entries, 0 to 352286
Data columns (total 66 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   player_name            352287 non-null  object        
 1   date                   352287 non-null  datetime64[ns]
 2   minute                 352287 non-null  datetime64[ns]
 3   minute_idx             352287 non-null  int64         
 4   lat_mean               352287 non-null  float64       
 5   lat_std                352287 non-null  float64       
 6   lat_min                352287 non-null  float64       
 7   lat_max                352287 non-null  float64       
 8   lon_mean               352287 non-null  float64       
 9   lon_std                352287 non-null  float64       
 10  lon_min                352287 non-null  float64       
 11  lon_max                352287 non-null  float64       
 12  speed_mean             352287 non-null  floa

### Loading Master Session 2020

In [5]:
master_session_2020 = pd.read_csv('D:/PhD EMP/2020/master_session.csv').drop("Unnamed: 0", axis=1)

master_session_2020.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3515 entries, 0 to 3514
Data columns (total 46 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   player_name         3515 non-null   object 
 1   team                3515 non-null   object 
 2   session_id          3515 non-null   object 
 3   total_minutes       3515 non-null   int64  
 4   speed_mean_sess     3515 non-null   float64
 5   speed_sum_sess      3515 non-null   float64
 6   hr_mean_sess        3515 non-null   float64
 7   hr_sum_sess         3515 non-null   float64
 8   inst_acc_mean_sess  3515 non-null   float64
 9   inst_acc_sum_sess   3515 non-null   float64
 10  hacc_mean_sess      3515 non-null   float64
 11  hacc_sum_sess       3515 non-null   float64
 12  accl_x_mean_sess    3515 non-null   float64
 13  accl_x_sum_sess     3515 non-null   float64
 14  accl_y_mean_sess    3515 non-null   float64
 15  accl_y_sum_sess     3515 non-null   float64
 16  accl_z

### Creating Modelling Dataset

In [6]:
### Joning minute level and master_session
final_df = minute_2020_data.merge(
    master_session_2020[
        [
            "player_name","session_id",
            "ctl28","ctl42","daily_load","weekly_load",
            "acwr","monotony","strain",
            "sleep_duration","sleep_quality",
            "fatigue","mood","readiness","soreness","stress",
            "injury","illness"
        ]
    ],
    on=["player_name","session_id"],
    how="left"
)

### Creating Feature Groups

In [7]:
id_cols = ["player_name", "session_id", "date", "minute", "minute_idx", "team"]
target_col = "injury"

feature_cols_baseline = [
    "ctl28", "ctl42", "daily_load", "weekly_load", "acwr", "monotony", "strain",
    "sleep_duration", "sleep_quality", "fatigue", "mood", "readiness", "soreness", "stress",
    "minute_idx",
    "speed_mean", "speed_std", "speed_max",
    "heart_rate_mean", "heart_rate_std", "heart_rate_max",
    "hacc_mean", "hacc_std", "hacc_max",
    "inst_acc_impulse_mean", "inst_acc_impulse_std", "inst_acc_impulse_max",
    "accl_x_std", "accl_y_std", "accl_z_std",
    "gyro_x_std", "gyro_y_std", "gyro_z_std"
]

### Inspecting Missingness

In [8]:
missing_summary = final_df[feature_cols_baseline].isna().mean().sort_values(ascending=False)
print(missing_summary)

sleep_quality            0.506740
sleep_duration           0.506740
fatigue                  0.500879
readiness                0.500586
stress                   0.500368
soreness                 0.500368
mood                     0.500368
ctl42                    0.290930
ctl28                    0.290930
strain                   0.290930
monotony                 0.290930
acwr                     0.290930
weekly_load              0.290930
daily_load               0.290930
hacc_max                 0.000000
gyro_y_std               0.000000
gyro_x_std               0.000000
accl_z_std               0.000000
accl_y_std               0.000000
accl_x_std               0.000000
inst_acc_impulse_max     0.000000
inst_acc_impulse_std     0.000000
inst_acc_impulse_mean    0.000000
speed_std                0.000000
hacc_std                 0.000000
hacc_mean                0.000000
heart_rate_max           0.000000
heart_rate_std           0.000000
heart_rate_mean          0.000000
speed_max     

In [9]:
wellness_cols = [
    "sleep_duration","sleep_quality","fatigue",
    "mood","readiness","soreness","stress"
]

for col in wellness_cols:
    final_df[col + "_missing"] = final_df[col].isna().astype(int)
    final_df[col] = final_df[col].fillna(final_df[col].median())

load_cols = [
    "ctl28","ctl42","daily_load","weekly_load",
    "acwr","monotony","strain"
]

for col in load_cols:
    final_df[col + "_missing"] = final_df[col].isna().astype(int)
    final_df[col] = final_df[col].fillna(final_df[col].median())

In [10]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 352287 entries, 0 to 352286
Data columns (total 96 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   player_name             352287 non-null  object        
 1   date                    352287 non-null  datetime64[ns]
 2   minute                  352287 non-null  datetime64[ns]
 3   minute_idx              352287 non-null  int64         
 4   lat_mean                352287 non-null  float64       
 5   lat_std                 352287 non-null  float64       
 6   lat_min                 352287 non-null  float64       
 7   lat_max                 352287 non-null  float64       
 8   lon_mean                352287 non-null  float64       
 9   lon_std                 352287 non-null  float64       
 10  lon_min                 352287 non-null  float64       
 11  lon_max                 352287 non-null  float64       
 12  speed_mean              352287

Handling Missing Values

Κατά την προεπεξεργασία των δεδομένων εντοπίστηκαν missing values κυρίως στις μεταβλητές που σχετίζονται με:

wellness metrics (π.χ. sleep, fatigue, stress)

training load metrics (π.χ. daily load, ACWR, strain)

Αντί να αφαιρεθούν οι εγγραφές, εφαρμόστηκε στρατηγική imputation, ώστε να διατηρηθούν όσο το δυνατόν περισσότερα δεδομένα.

Μέθοδος αντιμετώπισης

Η διαχείριση των missing values έγινε σε δύο βήματα:

1. Δημιουργία Missing Indicators

Για κάθε μεταβλητή που περιείχε missing values δημιουργήθηκε μια νέα δυαδική μεταβλητή (_missing), η οποία δείχνει αν η αρχική τιμή ήταν missing.

Η νέα μεταβλητή παίρνει τιμές:

1 → αν η αρχική τιμή ήταν missing

0 → αν η τιμή ήταν διαθέσιμη

Παράδειγμα:

sleep_duration_missing

Αυτό επιτρέπει στο μοντέλο να διατηρήσει πληροφορία για την αρχική απουσία δεδομένων.

2. Median Imputation

Στη συνέχεια οι missing τιμές αντικαταστάθηκαν με τη διάμεσο (median) της αντίστοιχης μεταβλητής:

final_df[col] = final_df[col].fillna(final_df[col].median())

Η median επιλέχθηκε επειδή:

είναι ανθεκτική σε outliers

διατηρεί καλύτερα την κατανομή των δεδομένων σε σχέση με τον μέσο όρο.

Μεταβλητές που επηρεάστηκαν

Η διαδικασία εφαρμόστηκε σε δύο ομάδες χαρακτηριστικών:

Wellness variables

sleep_duration

sleep_quality

fatigue

mood

readiness

soreness

stress

Training load variables

ctl28

ctl42

daily_load

weekly_load

acwr

monotony

strain

Αποτέλεσμα

Μετά τη διαδικασία:

όλες οι missing τιμές αντικαταστάθηκαν με τη median της κάθε μεταβλητής,

δημιουργήθηκαν νέες μεταβλητές που καταγράφουν την ύπαρξη missing τιμών,

το dataset έγινε πλήρες και κατάλληλο για χρήση σε μοντέλα μηχανικής μάθησης.

Τέλος, πραγματοποιήθηκε έλεγχος ώστε να επιβεβαιωθεί ότι δεν υπάρχουν πλέον missing τιμές στα χαρακτηριστικά του μοντέλου.

### Final Dataset Creation

In [11]:
target_col = ["injury"]
time_feature = ["minute_idx"]
speed_features = [
    "speed_mean",
    "speed_std",
    "speed_min",
    "speed_max"
]
heart_features = [
    "heart_rate_mean",
    "heart_rate_std",
    "heart_rate_min",
    "heart_rate_max"
]
hacc_features = [
    "hacc_mean",
    "hacc_std",
    "hacc_min",
    "hacc_max"
]
gps_quality_features = [
    "hdop_mean",
    "hdop_std",
    "hdop_min",
    "hdop_max",
    "signal_quality_mean",
    "signal_quality_std",
    "signal_quality_min",
    "signal_quality_max",
    "num_satellites_mean",
    "num_satellites_std",
    "num_satellites_min",
    "num_satellites_max"
]
impulse_features = [
    "inst_acc_impulse_mean",
    "inst_acc_impulse_std",
    "inst_acc_impulse_min",
    "inst_acc_impulse_max"
]
accel_features = [
    "accl_x_mean","accl_x_std","accl_x_min","accl_x_max",
    "accl_y_mean","accl_y_std","accl_y_min","accl_y_max",
    "accl_z_mean","accl_z_std","accl_z_min","accl_z_max"
]
gyro_features = [
    "gyro_x_mean","gyro_x_std","gyro_x_min","gyro_x_max",
    "gyro_y_mean","gyro_y_std","gyro_y_min","gyro_y_max",
    "gyro_z_mean","gyro_z_std","gyro_z_min","gyro_z_max"
]
load_features = [
    "ctl28",
    "ctl42",
    "daily_load",
    "weekly_load",
    "acwr",
    "monotony",
    "strain"
]
wellness_features = [
    "sleep_duration",
    "sleep_quality",
    "fatigue",
    "mood",
    "readiness",
    "soreness",
    "stress"
]

indicator_features = ['session_id', 'player_name']

### Final Features List
feature_cols = (
    time_feature +
    speed_features +
    heart_features +
    hacc_features +
    impulse_features +
    indicator_features +
    accel_features +
    gyro_features +
    load_features +
    wellness_features
)

model_df = final_df[feature_cols + target_col]

### Final Model Features Saving

In [12]:
model_df.to_csv('D:/PhD EMP/2020/modelling_features_df.csv')

Κρατήσαμε τις μεταβλητές που σχετίζονται άμεσα με φυσιολογικό φορτίο, μηχανικό φορτίο κίνησης, ιστορικό προπονητικού φορτίου και κατάσταση ευεξίας του αθλητή, επειδή αυτές έχουν τεκμηριωμένη σχέση με τον κίνδυνο τραυματισμού και είναι διαθέσιμες πριν ή μέχρι το minute 𝑥

άρα μπορούν να χρησιμοποιηθούν για να εκτιμηθεί η πιθανότητα ένα session να καταλήξει σε injury. 

Συγκεκριμένα, τα heart rate features αποτυπώνουν το εσωτερικό φυσιολογικό φορτίο, τα speed, acceleration και impulse metrics εκφράζουν το εξωτερικό μηχανικό φορτίο, τα IMU variance features περιγράφουν νευρομυϊκή σταθερότητα και κινηματική μεταβλητότητα, ενώ τα training load και wellness metrics παρέχουν πληροφορία για σωρευτική κόπωση και κατάσταση αποκατάστασης πριν το session. 

Αντίθετα, αφαιρέσαμε μεταβλητές που δεν έχουν άμεση βιομηχανική ή φυσιολογική σχέση με τον τραυματισμό ή αφορούν κυρίως την ποιότητα του αισθητήρα (π.χ. GPS coordinates, signal quality, satellites), καθώς και χαρακτηριστικά με χαμηλή πληροφοριακή αξία ή υψηλό θόρυβο (π.χ. πολλά minimum metrics), ώστε να μειωθεί η διάσταση του προβλήματος και να αποφευχθεί η εισαγωγή μη χρήσιμου σήματος στο μοντέλο.

In [13]:
model_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 352287 entries, 0 to 352286
Data columns (total 58 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   minute_idx             352287 non-null  int64  
 1   speed_mean             352287 non-null  float64
 2   speed_std              352287 non-null  float64
 3   speed_min              352287 non-null  float64
 4   speed_max              352287 non-null  float64
 5   heart_rate_mean        352287 non-null  float64
 6   heart_rate_std         352287 non-null  float64
 7   heart_rate_min         352287 non-null  float64
 8   heart_rate_max         352287 non-null  float64
 9   hacc_mean              352287 non-null  float64
 10  hacc_std               352287 non-null  float64
 11  hacc_min               352287 non-null  float64
 12  hacc_max               352287 non-null  float64
 13  inst_acc_impulse_mean  352287 non-null  float64
 14  inst_acc_impulse_std   352287 non-nu

### Creating Data up to minute X

In [14]:
# σωστό sorting
model_df = model_df.sort_values(
    ["player_name", "session_id", "minute_idx"]
).reset_index(drop=True)

# σωστό grouping
g = model_df.groupby(["player_name", "session_id"])


signals = [
    "speed_mean",
    "heart_rate_mean",
    "hacc_mean",
    "inst_acc_impulse_mean",
    "accl_x_std",
    "accl_y_std",
    "accl_z_std",
    "gyro_x_std",
    "gyro_y_std",
    "gyro_z_std"
]

for col in signals:

    # cumulative mean
    model_df[f"{col}_cum_mean"] = (
        g[col]
        .expanding()
        .mean()
        .reset_index(level=[0,1], drop=True)
    )

    # cumulative max
    model_df[f"{col}_cum_max"] = g[col].cummax()

    # cumulative std
    model_df[f"{col}_cum_std"] = (
        g[col]
        .expanding()
        .std()
        .reset_index(level=[0,1], drop=True)
    )

model_df = model_df.fillna(0)

print("Cumulative features created.")
print(model_df.shape)

Cumulative features created.
(352287, 88)


In [15]:
# =========================================
# DYNAMIC FEATURES (CRITICAL FOR TOP 1%)
# =========================================

# --- ΔΙΑΛΕΓΟΥΜΕ signals για dynamics ---
dynamic_signals = [
    "speed_mean",
    "heart_rate_mean",
    "hacc_mean",
    "inst_acc_impulse_mean",
    "accl_x_std",
    "accl_y_std",
    "accl_z_std",
    "gyro_x_std",
    "gyro_y_std",
    "gyro_z_std"
]

g = model_df.groupby(["player_name", "session_id"])

# =========================================
# 1️⃣ DELTAS (momentum)
# =========================================
for col in dynamic_signals:
    model_df[f"{col}_delta"] = g[col].diff()

# =========================================
# 2️⃣ ROLLING WINDOWS (recent state)
# =========================================
window = 5  # μπορείς να δοκιμάσεις και 3 ή 10

for col in dynamic_signals:
    model_df[f"{col}_roll_mean"] = (
        g[col]
        .rolling(window)
        .mean()
        .reset_index(level=[0,1], drop=True)
    )
    
    model_df[f"{col}_roll_std"] = (
        g[col]
        .rolling(window)
        .std()
        .reset_index(level=[0,1], drop=True)
    )

# =========================================
# 3️⃣ SPIKES (acute load)
# =========================================

# Heart rate spike
model_df["hr_spike"] = model_df["heart_rate_max"] - model_df["heart_rate_mean"]

# Speed spike
model_df["speed_spike"] = model_df["speed_max"] - model_df["speed_mean"]

# Impulse spike
model_df["impulse_spike"] = (
    model_df["inst_acc_impulse_max"] - model_df["inst_acc_impulse_mean"]
)

# =========================================
# 4️⃣ FATIGUE TREND (IMPORTANT)
# =========================================

model_df["hr_trend"] = (
    g["heart_rate_mean"]
    .expanding()
    .mean()
    .reset_index(level=[0,1], drop=True)
)

# =========================================
# CLEAN NaNs
# =========================================
model_df = model_df.fillna(0)

print("Dynamic features created.")
print(model_df.shape)

Dynamic features created.
(352287, 122)


In [16]:
# Identifiers (δεν χρησιμοποιούνται ως predictors αλλά χρειάζονται για split)
id_cols = [
    "session_id",
    "minute_idx",
    "player_name"
]

# Pre-session load features
load_features = [
    "ctl28",
    "ctl42",
    "daily_load",
    "weekly_load",
    "acwr",
    "monotony",
    "strain"
]

# Wellness features
wellness_features = [
    "sleep_duration",
    "sleep_quality",
    "fatigue",
    "mood",
    "readiness",
    "soreness",
    "stress"
]

# Cumulative biomechanical features (data up to minute x)
cumulative_features = [

    # Speed
    "speed_mean_cum_mean",
    "speed_mean_cum_max",
    "speed_mean_cum_std",

    # Heart rate
    "heart_rate_mean_cum_mean",
    "heart_rate_mean_cum_max",
    "heart_rate_mean_cum_std",

    # Horizontal acceleration
    "hacc_mean_cum_mean",
    "hacc_mean_cum_max",
    "hacc_mean_cum_std",

    # Acceleration impulse
    "inst_acc_impulse_mean_cum_mean",
    "inst_acc_impulse_mean_cum_max",
    "inst_acc_impulse_mean_cum_std",

    # Accelerometer variability
    "accl_x_std_cum_mean",
    "accl_x_std_cum_max",
    "accl_x_std_cum_std",

    "accl_y_std_cum_mean",
    "accl_y_std_cum_max",
    "accl_y_std_cum_std",

    "accl_z_std_cum_mean",
    "accl_z_std_cum_max",
    "accl_z_std_cum_std",

    # Gyroscope variability
    "gyro_x_std_cum_mean",
    "gyro_x_std_cum_max",
    "gyro_x_std_cum_std",

    "gyro_y_std_cum_mean",
    "gyro_y_std_cum_max",
    "gyro_y_std_cum_std",

    "gyro_z_std_cum_mean",
    "gyro_z_std_cum_max",
    "gyro_z_std_cum_std"
]

dynamic_features = []

# DELTAS
for col in signals:
    dynamic_features.append(f"{col}_delta")

# ROLLING
for col in signals:
    dynamic_features.append(f"{col}_roll_mean")
    dynamic_features.append(f"{col}_roll_std")

# SPIKES
dynamic_features += [
    "hr_spike",
    "speed_spike",
    "impulse_spike",
    "hr_trend"
]

# Target
target = ["injury"]

# Όλα τα columns που κρατάμε
model_columns = (
    id_cols +
    load_features +
    wellness_features +
    cumulative_features +
    dynamic_features + 
    target
)

# Δημιουργία modelling dataset
df_model = model_df[model_columns].copy()

print("Final modelling dataset shape:", df_model.shape)

Final modelling dataset shape: (352287, 82)


In [17]:
df_model.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 352287 entries, 0 to 352286
Data columns (total 82 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   session_id                       352287 non-null  object 
 1   minute_idx                       352287 non-null  int64  
 2   player_name                      352287 non-null  object 
 3   ctl28                            352287 non-null  float64
 4   ctl42                            352287 non-null  float64
 5   daily_load                       352287 non-null  float64
 6   weekly_load                      352287 non-null  float64
 7   acwr                             352287 non-null  float64
 8   monotony                         352287 non-null  float64
 9   strain                           352287 non-null  float64
 10  sleep_duration                   352287 non-null  float64
 11  sleep_quality                    352287 non-null  float64
 12  fa

* Cumulative mean, maximum and variability statistics were computed for the main load-related signals (speed, heart rate and acceleration). For inertial measurement unit signals, which already represent movement variability, cumulative averages were used to capture the evolution of movement instability throughout the session.

### Writting model_df

In [18]:
df_model.to_csv('D:/PhD EMP/2020/model_df.csv')